# Preprocessing ablation for sentiment training (v2)

**Why this rewrite.** The first version tested single-axis variants and produced MAE differences of ~0.005 on a single 10k split — within sampling noise, so unactionable. This version uses (a) coarser tiers that vary multiple axes at once, (b) bootstrap CIs to separate signal from noise, (c) a 20k sample to halve the noise floor, (d) self-contained cells.

Sections:
- **§1** Six preprocessing tiers from `raw_passthrough` to `max_intervention`.
- **§2** Bootstrap confidence intervals via N=10 stratified resamples per tier.
- **§3** Conflict-label dedup test using full-corpus conflict detection.
- **Conclusion**.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()
if current_dir.name == "baselines":
    os.chdir("..")
project_root = str(Path.cwd().resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import spacy
from data_analysis.utils.text_analysis import sentiment_evaluator
from data_analysis.utils.text_preprocessing import TextPreprocessor

In [ ]:
DEBUG_PATH = "data/splitlang/DEBUG/train.csv"
SAMPLE_PER_LABEL = 4000   # 4k * 5 labels = 20k stratified sample
RANDOM_STATE = 42
N_BOOTSTRAP = 10

debug = pd.read_csv(DEBUG_PATH)
sample = (
    debug.groupby("label")
         .sample(n=SAMPLE_PER_LABEL, random_state=RANDOM_STATE)
         .reset_index(drop=True)
)
print(f"corpus: {len(debug):,} rows")
print(f"sample: {len(sample):,} rows   |   languages: {sample['_lang'].value_counts().to_dict()}")

print(f"spaCy GPU available  : {spacy.prefer_gpu()}")
SPACY_DISABLE = ["parser", "lemmatizer"]

def _load(model):
    try:
        return spacy.load(model, disable=SPACY_DISABLE)
    except OSError:
        spacy.cli.download(model)
        return spacy.load(model, disable=SPACY_DISABLE)

nlp_by_lang = {"EN": _load("en_core_web_sm"), "DE": _load("de_core_news_sm")}
print(f"spaCy pipeline: {nlp_by_lang['EN'].pipe_names}")

## 1. Coarse preprocessing tiers

Six intensity levels. Each tier varies multiple axes at once, so the BOW classifier sees a meaningfully different input vs. its neighbours.

| tier | what it does |
|---|---|
| `raw_passthrough` | nothing (TF-IDF lowercases internally). The floor. |
| `minimal` | strip HTML + URLs + cap repeated punct at 3. Cheap default. |
| `moderate` | minimal + strip emojis. |
| `our_default` | moderate + strip entity descriptors. The current `baseline` config. |
| `aggressive` | our_default + cap punct at 1 + replace entities with `<BRAND>` token. |
| `max_intervention` | aggressive + replace emojis with `<POS_EMOJI>` / `<NEG_EMOJI>` / `<NEU_EMOJI>` tokens. |

In [ ]:
TIERS = [
    {"name": "raw_passthrough",  "strip_html": False, "remove_urls": False, "emoji_strategy": "keep",          "punct_max_repeat": None, "entity_strategy": "keep"},
    {"name": "minimal",          "strip_html": True,  "remove_urls": True,  "emoji_strategy": "keep",          "punct_max_repeat": 3,    "entity_strategy": "keep"},
    {"name": "moderate",         "strip_html": True,  "remove_urls": True,  "emoji_strategy": "strip",         "punct_max_repeat": 3,    "entity_strategy": "keep"},
    {"name": "our_default",      "strip_html": True,  "remove_urls": True,  "emoji_strategy": "strip",         "punct_max_repeat": 3,    "entity_strategy": "strip_descriptors"},
    {"name": "aggressive",       "strip_html": True,  "remove_urls": True,  "emoji_strategy": "strip",         "punct_max_repeat": 1,    "entity_strategy": "replace_with_token"},
    {"name": "max_intervention", "strip_html": True,  "remove_urls": True,  "emoji_strategy": "pos_neg_token", "punct_max_repeat": 1,    "entity_strategy": "replace_with_token"},
]
BERT_STYLE = {"lowercase": False, "final_tokenize": False}
for t in TIERS:
    t.update(BERT_STYLE)
TIER_NAMES = [t["name"] for t in TIERS]

In [ ]:
tier_texts: dict[str, list[str]] = {}
for tier in TIERS:
    name = tier["name"]
    pre = TextPreprocessor(**{k: v for k, v in tier.items() if k != "name"})
    cleaned = [""] * len(sample)
    t0 = time.time()
    for lang, sub in sample.groupby("_lang"):
        out = pre.preprocess_batch(
            sub["sentence"].tolist(),
            nlp=nlp_by_lang.get(lang),
            lang=lang.lower(),
        )
        for i, idx in enumerate(sub.index):
            cleaned[idx] = out[i]
    tier_texts[name] = cleaned
    n_chars = sum(len(c) for c in cleaned)
    print(f"  [{name:18s}] preprocessed in {time.time() - t0:5.1f}s   total chars: {n_chars:,}")

In [ ]:
labels = sample["label"].tolist()
rows = []
for tier in TIERS:
    name = tier["name"]
    m = sentiment_evaluator(tier_texts[name], labels)
    rows.append({"tier": name, **m})
    print(f"  [{name:18s}] mae={m['mae']:.4f}  score={m['score']:.4f}  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}")

tier_results = pd.DataFrame(rows)
tier_results.to_csv("baselines/outputs_v2/our_preprocessing_tiers.csv", index=False)
display(tier_results[["tier", "mae", "score", "accuracy", "f1_macro"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
results_ord = tier_results.set_index("tier").loc[TIER_NAMES]
default_idx = TIER_NAMES.index("our_default")
colors = ["#4C72B0" if i == default_idx else "#CCCCCC" for i in range(len(TIER_NAMES))]

for ax, metric, title, lower_is_better in [
    (axes[0], "mae",      "BOW dev MAE per tier",       True),
    (axes[1], "f1_macro", "BOW dev macro-F1 per tier",  False),
]:
    bars = ax.barh(TIER_NAMES, results_ord[metric], color=colors, edgecolor="white")
    ax.axvline(results_ord.loc["our_default", metric], color="#4C72B0",
               linestyle="--", alpha=0.5, label="our_default")
    ax.set_xlabel(f"{metric}  ({'lower' if lower_is_better else 'higher'} is better)")
    ax.set_title(title)
    for bar, v in zip(bars, results_ord[metric]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=9)
    ax.invert_yaxis()
    ax.legend()

plt.tight_layout()
plt.savefig("data_analysis/img/preprocessing_tiers.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Bootstrap CIs — is the difference real?

A single 80/20 split has ~0.005 MAE noise floor on a 20k sample, which can swallow real preprocessing effects. We re-evaluate each tier on **N=10 stratified resamples** (different `random_state`) and report the MAE distribution. Overlapping boxes = statistically indistinguishable on this sample size.

In [ ]:
boot_rows = []
for tier in TIERS:
    name = tier["name"]
    for seed in range(N_BOOTSTRAP):
        m = sentiment_evaluator(tier_texts[name], labels, random_state=1000 + seed)
        boot_rows.append({"tier": name, "seed": seed, "mae": m["mae"], "f1_macro": m["f1_macro"], "score": m["score"]})
    print(f"  [{name:18s}] {N_BOOTSTRAP} resamples done")

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv("baselines/outputs_v2/our_preprocessing_tiers_bootstrap.csv", index=False)

summary = (
    boot_df.groupby("tier")["mae"]
           .agg(mean="mean", std="std",
                ci_lo=lambda s: float(np.quantile(s, 0.025)),
                ci_hi=lambda s: float(np.quantile(s, 0.975)))
           .round(4)
           .reindex(TIER_NAMES)
)
display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
data = [boot_df[boot_df["tier"] == t]["mae"].values for t in TIER_NAMES]

bp = ax.boxplot(data, positions=range(len(TIER_NAMES)), widths=0.6,
                patch_artist=True, showmeans=True,
                medianprops={"color": "black", "linewidth": 1.2},
                meanprops={"marker": "D", "markerfacecolor": "#4C72B0",
                           "markeredgecolor": "black", "markersize": 7})
for patch in bp["boxes"]:
    patch.set_facecolor("#E8E8E8")
    patch.set_edgecolor("#888")
ax.set_xticks(range(len(TIER_NAMES)))
ax.set_xticklabels(TIER_NAMES, rotation=20, ha="right")
ax.set_ylabel(f"validation MAE  (n = {N_BOOTSTRAP} resamples)")
ax.set_title("Per-tier MAE distribution — overlapping boxes are indistinguishable")
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.axhline(boot_df[boot_df["tier"] == "our_default"]["mae"].mean(),
           color="#4C72B0", linestyle="--", alpha=0.5, label="our_default mean")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("data_analysis/img/preprocessing_tiers_bootstrap.png", dpi=120, bbox_inches="tight")
plt.show()

default_maes = boot_df[boot_df["tier"] == "our_default"].set_index("seed")["mae"]
diffs = []
for tier in TIER_NAMES:
    if tier == "our_default":
        continue
    other = boot_df[boot_df["tier"] == tier].set_index("seed")["mae"]
    delta = (other - default_maes).values
    diffs.append({
        "tier":       tier,
        "mean_delta": float(delta.mean()),
        "std_delta":  float(delta.std()),
        "ci_lo":      float(np.quantile(delta, 0.025)),
        "ci_hi":      float(np.quantile(delta, 0.975)),
        "verdict":    "better than default" if delta.mean() < 0 and np.quantile(delta, 0.975) < 0 else
                       "worse than default"  if delta.mean() > 0 and np.quantile(delta, 0.025) > 0 else
                       "indistinguishable",
    })
diff_df = pd.DataFrame(diffs).round(4)
print("\nMAE delta vs our_default (negative = better):")
display(diff_df)

## 3. Conflict-label dedup impact

Youssef found 27 distinct preprocessed texts in the **full 252k corpus** that appear with multiple labels — genuine label noise. Rather than rely on the 20k sample (which contains only ~1-2 of those rows by chance), we use the full corpus to identify the conflict groups and remove any sample row whose preprocessed text matches one of them.

Bootstrap CI included — small effects need it.

In [ ]:
# Use the splitlang `_preprocessed_text` column (computed on the full corpus
# by the language splitter) to identify label-conflicting groups. This is a
# different cleaner than `our_default` but a reasonable proxy for finding noise.
labels_per_text = debug.groupby("_preprocessed_text")["label"].nunique()
conflict_full = set(labels_per_text[labels_per_text > 1].index)
print(f"full-corpus conflict groups: {len(conflict_full)}")
print(f"full-corpus rows in those groups: {int(debug['_preprocessed_text'].isin(conflict_full).sum()):,}")

# Identify which rows in our 20k sample sit in a conflict group.
sample_in_conflict = sample["_preprocessed_text"].isin(conflict_full)
print(f"sample rows in a conflict group  : {int(sample_in_conflict.sum())}")

In [ ]:
default_clean = tier_texts["our_default"]
labels_arr = np.asarray(labels)
clean_arr = np.asarray(default_clean, dtype=object)
keep_mask = (~sample_in_conflict).values

rows_all, rows_clean = [], []
for seed in range(N_BOOTSTRAP):
    rows_all.append(sentiment_evaluator(clean_arr.tolist(), labels_arr.tolist(),
                                         random_state=2000 + seed)["mae"])
    rows_clean.append(sentiment_evaluator(clean_arr[keep_mask].tolist(),
                                           labels_arr[keep_mask].tolist(),
                                           random_state=2000 + seed)["mae"])
rows_all = np.array(rows_all)
rows_clean = np.array(rows_clean)
delta = rows_clean - rows_all

print(f"sample rows total              : {len(default_clean):,}")
print(f"sample rows kept after dedup   : {int(keep_mask.sum()):,}  (removed {int((~keep_mask).sum())})")
print(f"MAE all rows           : {rows_all.mean():.4f} +/- {rows_all.std():.4f}")
print(f"MAE conflict-removed   : {rows_clean.mean():.4f} +/- {rows_clean.std():.4f}")
print(f"delta (negative=better): {delta.mean():+.4f}   95% CI [{np.quantile(delta, 0.025):+.4f}, {np.quantile(delta, 0.975):+.4f}]")
if delta.mean() < 0 and np.quantile(delta, 0.975) < 0:
    print("verdict: dedup HELPS (significant)")
elif delta.mean() > 0 and np.quantile(delta, 0.025) > 0:
    print("verdict: dedup HURTS (significant)")
else:
    print("verdict: dedup has no detectable effect at this sample size")

## Conclusion — what to use for retraining

Read the **§2 verdict table** (the `diff_df`) to fill in the answers:

| question | how to read it |
|---|---|
| Is any tier *significantly* better than `our_default`? | look for rows with `verdict == "better than default"` |
| Is `raw_passthrough` *significantly* worse? | tells us whether preprocessing matters at all for BOW |
| Does dedup help? | §3 verdict line |

**Decision rule:**
1. If §2 shows **all tiers indistinguishable**: BOW is robust to preprocessing on this corpus. Pick the cheapest tier (`minimal`) for the BOW baseline. For BERT, default to `our_default` since BERT should benefit more from cleaning than BOW does.
2. If §2 shows **a clear winner**: use it for both BOW and BERT regardless of cost.
3. If §3 dedup shows a **significant gain**: run dedup as a final step on the training corpus before fitting either model.

All result CSVs are in `baselines/outputs_v2/our_preprocessing_*.csv`.